In [1]:
!pip install python-dotenv

In [2]:
%%writefile .env
minha_chave=GKe752e73e675cc700c0eb72f3
secret_key=0d28e7f60c63d7c149ea5f4378530f91fca9ab2e57c6fbb970bcc12c98c33252

Overwriting .env


In [3]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession

# Carrega na memória as variáveis declaradas no arquivo oculto
load_dotenv()
access_key = os.getenv('minha_chave')
secret_key = os.getenv('secret_key')

# Instancia a sessão do Spark injetando dinamicamente os pacotes do Kafka e S3A AWS Hadoop
spark = SparkSession.builder \
    .appName("Security-DataLake-Pipeline") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.apache.hadoop:hadoop-aws:3.3.4") \
    .getOrCreate()

# Configuração detalhada do driver de mapeamento de objetos para o endpoint do Garage
sc = spark.sparkContext
sc._jsc.hadoopConfiguration().set("fs.s3a.access.key", access_key)
sc._jsc.hadoopConfiguration().set("fs.s3a.secret.key", secret_key)
sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://garage:3900")
sc._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
sc._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
sc._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")

# Parâmetros de compatibilidade regional do Garage
sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint.region", "garage")
sc._jsc.hadoopConfiguration().set("fs.s3a.region", "garage")
sc._jsc.hadoopConfiguration().set("fs.s3a.change.detection.mode", "none")
sc._jsc.hadoopConfiguration().set("fs.s3a.change.detection.source", "none")

# ⚡ TUNING DE PERFORMANCE: ACELERAÇÃO DE ESCRITA NO OBJECT STORAGE LOCAL
# 1. Altera o algoritmo de commit para escrita direta no destino final
sc._jsc.hadoopConfiguration().set("mapreduce.fileoutputcommitter.algorithm.version", "2")

# 2. Mantém os marcadores de diretório, eliminando o loop de requisições DELETE (Crucial para o Garage!)
sc._jsc.hadoopConfiguration().set("fs.s3a.directory.marker.retention", "keep")

# 3. Ativa o pipe de upload rápido via buffers de memória ram do container
sc._jsc.hadoopConfiguration().set("fs.s3a.fast.upload", "true")
sc._jsc.hadoopConfiguration().set("fs.s3a.fast.upload.buffer", "disk")

print("🚀 Sessão Spark tunada e otimizada para o Garage HQ!")

🚀 Sessão Spark tunada e otimizada para o Garage HQ!


In [4]:
# Massa de dados de teste contendo metadados de incidentes
dados_teste = [
    ("VLAN_30", "192.168.30.99", "Mirai Port Scan Detectado", "ALTA"),
    ("VLAN_40", "192.168.40.12", "Tentativa Bruteforce SSH", "MEDIA")
]
colunas = ["origem_vlan", "ip_origem", "evento", "severidade"]
df_teste = spark.createDataFrame(dados_teste, schema=colunas)

try:
    # Persiste em formato binário otimizado colunar Parquet
    df_teste.write.format("parquet").mode("overwrite").save("s3a://meu-data-lake/teste_alertas/")
    print("✨ Sucesso! O Spark conseguiu autenticar, gravar e fechar pacotes no Garage HQ.")
    
    # Valida a leitura reversa do dado persistido
    spark.read.parquet("s3a://meu-data-lake/teste_alertas/").show()
except Exception as e:
    print(f"❌ Falha crítica de privilégio ou IO no Object Storage: {e}")

✨ Sucesso! O Spark conseguiu autenticar, gravar e fechar pacotes no Garage HQ.
+-----------+-------------+--------------------+----------+
|origem_vlan|    ip_origem|              evento|severidade|
+-----------+-------------+--------------------+----------+
|    VLAN_30|192.168.30.99|Mirai Port Scan D...|      ALTA|
|    VLAN_40|192.168.40.12|Tentativa Brutefo...|     MEDIA|
+-----------+-------------+--------------------+----------+



In [5]:
spark.read.parquet("s3a://meu-data-lake/teste_alertas/").show()

+-----------+-------------+--------------------+----------+
|origem_vlan|    ip_origem|              evento|severidade|
+-----------+-------------+--------------------+----------+
|    VLAN_30|192.168.30.99|Mirai Port Scan D...|      ALTA|
|    VLAN_40|192.168.40.12|Tentativa Brutefo...|     MEDIA|
+-----------+-------------+--------------------+----------+



In [6]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Mapeamento do Schema padronizado
log_schema = StructType([
    StructField("vlan", IntegerType(), True),
    StructField("src_ip", StringType(), True),
    StructField("dst_ip", StringType(), True),
    StructField("dst_port", IntegerType(), True),
    StructField("packets", IntegerType(), True),
    StructField("bytes", IntegerType(), True)
])

# Conexão de streaming contínuo ao cluster interno do Kafka
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "ipfix-network-flow") \
    .option("startingOffsets", "earliest") \
    .load()
# 🟢 AJUSTE: Mudado para 'earliest' para capturar os testes anteriores


# Conversão do binário de carga do Kafka em dados tipados estruturados
df_parsed = df_kafka.selectExpr("CAST(value AS STRING) as json_payload") \
    .select(from_json(col("json_payload"), log_schema).alias("data")) \
    .select("data.*") \
    .withColumn("ingestion_time", current_timestamp())

# Escrita incremental contínua salvando de forma otimizada e particionada por VLAN no Garage
query = df_parsed.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", "s3a://meu-data-lake/live_network_logs/") \
    .option("checkpointLocation", "s3a://meu-data-lake/checkpoints/logs_pipeline/") \
    .partitionBy("vlan") \
    .start()

print("🔥 Pipeline de streaming em tempo real ativado em segundo plano!")

🔥 Pipeline de streaming em tempo real ativado em segundo plano!


In [7]:
# 1. Verifica se a thread em segundo plano continua viva e ativa
print(f"O streaming está ativo? {query.isActive}")

# 2. Mostra as estatísticas da última leitura (quantas linhas ele processou por segundo)
import json
print(json.dumps(query.lastProgress, indent=2))

O streaming está ativo? True
null


In [8]:
spark.read.parquet("s3a://meu-data-lake/teste_alertas/").show()

+-----------+-------------+--------------------+----------+
|origem_vlan|    ip_origem|              evento|severidade|
+-----------+-------------+--------------------+----------+
|    VLAN_30|192.168.30.99|Mirai Port Scan D...|      ALTA|
|    VLAN_40|192.168.40.12|Tentativa Brutefo...|     MEDIA|
+-----------+-------------+--------------------+----------+



In [9]:
spark.read.parquet("s3a://meu-data-lake/live_network_logs/").show()

+-------------+------------+--------+-------+-----+--------------------+----+
|       src_ip|      dst_ip|dst_port|packets|bytes|      ingestion_time|vlan|
+-------------+------------+--------+-------+-----+--------------------+----+
|192.168.30.77|192.168.30.1|      23|    100| 1000|2026-06-29 23:02:...|  30|
|192.168.30.78|192.168.30.1|      23|    200| 2000|2026-06-29 23:02:...|  30|
|192.168.30.77|192.168.30.1|      23|    600|36000|2026-06-29 22:51:...|  30|
|192.168.30.77|192.168.30.1|      23|    600|36000|2026-06-29 22:52:...|  30|
|192.168.30.77|192.168.30.1|      23|    600|36000|2026-06-29 23:13:...|  30|
|192.168.30.77|192.168.30.1|      23|    600|36000|2026-06-29 22:53:...|1010|
|192.168.40.10|192.168.9.50|      80|    150|12500|2026-06-29 23:13:...|  40|
+-------------+------------+--------+-------+-----+--------------------+----+

